In [5]:
# !pip install scikit-optimize

### Logistic Regression

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

### 1. 사전 작업


In [7]:
# ADASYN 적용된 학습 데이터 로드
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")

print("shape:", train_df.shape)
print(train_df.head())
print("\ncolumns:")
print(train_df.columns.tolist())

# 실제 타깃 컬럼명
target_col = "Risk_Label"

# 전체 데이터(X)와 타깃(y) 분리
# 주의: X에 Risk_Label이 포함되면 데이터 누수가 생기므로 반드시 제거
X = train_df.drop(columns=[target_col]).copy()
y = train_df[target_col].astype(int)

print("\nX shape:", X.shape)
print("y distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True))

# 불균형 데이터 평가를 위한 G-Mean 함수
def gmean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return np.sqrt(recall * specificity)

shape: (3270, 11)
   NASDAQ_return(%)  Brent Crude Oil_return(%)  GJR_VaR_5_t1  \
0          0.593612                   0.545323      0.643782   
1          0.259590                   0.545323      0.662523   
2          0.758918                   0.545323      0.679774   
3          0.592042                   0.545323      0.696228   
4          0.610832                   0.545323      0.712514   

   Gold Spot_return(%)  KOSPI 200 Volume  KOSPI 200_DMI14  KOSPI 200_OG  \
0             0.488488          0.000817         0.782213      0.728036   
1             0.696235          0.000552         0.758401      0.655851   
2             0.535190          0.000633         0.721068      0.325430   
3             0.630347          0.000972         0.768348      0.662242   
4             0.657784          0.000917         0.791378      0.671566   

   Signal2_Buy  Signal2_Sell  VKOSPI_Close  Risk_Label  
0            0             0      0.633584           0  
1            0             0    

### 2. Valid 기반 파라미터 최적화

In [8]:
# =========================
# 2. Valid 기반 LR 하이퍼파라미터 + cutoff 동시 탐색
# =========================
import os
import numpy as np
import pandas as pd
import sklearn
from packaging import version

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    log_loss, roc_auc_score, average_precision_score
)

# -------------------------
# 1) 데이터 로드
# -------------------------
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")

target_col = "Risk_Label"

# X에 target이 들어가면 데이터 누수이므로 반드시 제거
X_train_raw = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].astype(int)

X_valid_raw = valid_df.drop(columns=[target_col]).copy()
y_valid = valid_df[target_col].astype(int)

# valid/test에서 컬럼 불일치가 있으면 0으로 채우지 말고 오류 내는 게 안전함
missing_in_valid = set(X_train_raw.columns) - set(X_valid_raw.columns)
extra_in_valid = set(X_valid_raw.columns) - set(X_train_raw.columns)

if missing_in_valid or extra_in_valid:
    raise ValueError(
        f"train-valid 컬럼 불일치\n"
        f"valid에 없는 train 컬럼: {missing_in_valid}\n"
        f"valid에만 있는 컬럼: {extra_in_valid}"
    )

X_valid_raw = X_valid_raw[X_train_raw.columns]

X_train_model = X_train_raw.values
X_valid_model = X_valid_raw.values
feature_names = X_train_raw.columns

print("Train shape:", X_train_model.shape)
print("Valid shape:", X_valid_model.shape)
print("\nTrain class distribution")
print(y_train.value_counts(normalize=True))
print("\nValid class distribution")
print(y_valid.value_counts(normalize=True))


# -------------------------
# 2) 평가 함수
# -------------------------
def get_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean = np.sqrt(rec * specificity)

    try:
        ll = log_loss(y_true, y_prob, labels=[0, 1])
    except Exception:
        ll = np.nan

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except Exception:
        pr_auc = np.nan

    return {
        "threshold": threshold,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "specificity": specificity,
        "f1": f1,
        "gmean": gmean,
        "log_loss": ll,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def make_threshold_grid(valid_prob):
    # 0.01 간격 cutoff + validation 확률 분위수 기반 cutoff를 함께 사용
    # 확률이 특정 구간에 몰려 있어도 후보 cutoff가 충분히 잡히도록 함
    return np.unique(np.r_[
        np.arange(0.01, 0.991, 0.01),
        np.quantile(valid_prob, np.linspace(0.01, 0.99, 99))
    ])


def find_best_cutoff(y_valid, valid_prob):
    threshold_history = []

    for thr in make_threshold_grid(valid_prob):
        m = get_binary_metrics(y_valid, valid_prob, threshold=thr)
        threshold_history.append({
            "threshold": float(thr),
            "valid_accuracy": m["accuracy"],
            "valid_precision": m["precision"],
            "valid_recall": m["recall"],
            "valid_specificity": m["specificity"],
            "valid_f1": m["f1"],
            "valid_gmean": m["gmean"],
            "valid_log_loss": m["log_loss"],
            "valid_roc_auc": m["roc_auc"],
            "valid_pr_auc": m["pr_auc"],
            "tn": m["tn"],
            "fp": m["fp"],
            "fn": m["fn"],
            "tp": m["tp"]
        })

    threshold_df = pd.DataFrame(threshold_history)

    # cutoff 선택 기준:
    # 1) valid G-Mean 최고값의 99% 이상인 cutoff 후보만 유지
    # 2) 그 후보들 중 valid Precision 최대
    # 3) 동률이면 valid G-Mean → valid F1 → valid Recall → valid LogLoss 순서로 선택
    max_gmean = threshold_df["valid_gmean"].max()
    threshold_df_eligible = threshold_df[
        threshold_df["valid_gmean"] >= 0.99 * max_gmean
    ].copy()

    threshold_df_eligible = threshold_df_eligible.sort_values(
        by=["valid_precision", "valid_gmean", "valid_f1", "valid_recall", "valid_log_loss"],
        ascending=[False, False, False, False, True]
    ).reset_index(drop=True)

    threshold_df = threshold_df.sort_values(
        by=["valid_gmean", "valid_precision", "valid_f1", "valid_recall", "valid_log_loss"],
        ascending=[False, False, False, False, True]
    ).reset_index(drop=True)

    return threshold_df_eligible.iloc[0], threshold_df


# -------------------------
# 3) sklearn 버전 호환 LR 생성 함수
# -------------------------
def make_lr(C, l1_ratio, max_iter=10000, random_state=1):
    kwargs = dict(
        solver="saga",
        C=C,
        l1_ratio=l1_ratio,
        max_iter=max_iter,
        random_state=random_state,
        tol=1e-4
    )

    # sklearn 1.8부터 penalty 경고가 뜨므로 버전별 처리
    if version.parse(sklearn.__version__) < version.parse("1.8"):
        kwargs["penalty"] = "elasticnet"

    return LogisticRegression(**kwargs)


# -------------------------
# 4) 1차 coarse grid search
#    C, l1_ratio, cutoff를 valid set에서 동시에 선택
# -------------------------
C_grid_1 = np.logspace(-4, 4, 33)
l1_grid_1 = np.round(np.linspace(0.0, 1.0, 21), 2)

search_history_1 = []
threshold_history_all_1 = []

print(f"\n[1차 탐색] LR 조합 수: {len(C_grid_1) * len(l1_grid_1)}")
print("각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.")
print("선택 기준: valid G-Mean 최고값의 99% 이상 후보 중 valid Precision 최대")

for C_val in C_grid_1:
    for l1_val in l1_grid_1:
        model_trial = make_lr(C=C_val, l1_ratio=l1_val)
        model_trial.fit(X_train_model, y_train)

        valid_prob = model_trial.predict_proba(X_valid_model)[:, 1]
        best_thr_row, threshold_df_tmp = find_best_cutoff(y_valid, valid_prob)

        search_history_1.append({
            "stage": "coarse",
            "C": C_val,
            "l1_ratio": l1_val,
            "best_cutoff": best_thr_row["threshold"],
            "valid_accuracy": best_thr_row["valid_accuracy"],
            "valid_precision": best_thr_row["valid_precision"],
            "valid_recall": best_thr_row["valid_recall"],
            "valid_specificity": best_thr_row["valid_specificity"],
            "valid_f1": best_thr_row["valid_f1"],
            "valid_gmean": best_thr_row["valid_gmean"],
            "valid_log_loss": best_thr_row["valid_log_loss"],
            "valid_roc_auc": best_thr_row["valid_roc_auc"],
            "valid_pr_auc": best_thr_row["valid_pr_auc"],
            "tn": best_thr_row["tn"],
            "fp": best_thr_row["fp"],
            "fn": best_thr_row["fn"],
            "tp": best_thr_row["tp"]
        })

        threshold_df_tmp = threshold_df_tmp.copy()
        threshold_df_tmp["stage"] = "coarse"
        threshold_df_tmp["C"] = C_val
        threshold_df_tmp["l1_ratio"] = l1_val
        threshold_history_all_1.append(threshold_df_tmp)

search_df_1 = pd.DataFrame(search_history_1)

max_gmean_1 = search_df_1["valid_gmean"].max()
search_df_1_eligible = search_df_1[
    search_df_1["valid_gmean"] >= 0.99 * max_gmean_1
].copy()

search_df_1 = search_df_1_eligible.sort_values(
    by=["valid_precision", "valid_gmean", "valid_f1", "valid_recall", "valid_log_loss"],
    ascending=[False, False, False, False, True]
).reset_index(drop=True)

print("\n[1차 탐색 상위 10개: G-Mean 99% 이상 후보 중 Precision 우선]")
print(search_df_1.head(10))

best_1 = search_df_1.iloc[0]
best_C_1 = float(best_1["C"])
best_l1_1 = float(best_1["l1_ratio"])

print("\n[1차 Best]")
print(f"C={best_C_1:.8f}, l1_ratio={best_l1_1:.4f}, cutoff={best_1['best_cutoff']:.4f}, "
      f"G-Mean={best_1['valid_gmean']:.4f}, Precision={best_1['valid_precision']:.4f}, "
      f"F1={best_1['valid_f1']:.4f}, Recall={best_1['valid_recall']:.4f}")

Train shape: (3270, 10)
Valid shape: (1438, 10)

Train class distribution
Risk_Label
0    0.500917
1    0.499083
Name: proportion, dtype: float64

Valid class distribution
Risk_Label
0    0.873435
1    0.126565
Name: proportion, dtype: float64

[1차 탐색] LR 조합 수: 693
각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.
선택 기준: valid G-Mean 최고값의 99% 이상 후보 중 valid Precision 최대

[1차 탐색 상위 10개: G-Mean 99% 이상 후보 중 Precision 우선]
    stage         C  l1_ratio  best_cutoff  valid_accuracy  valid_precision  \
0  coarse  1.000000      0.00     0.492223        0.707232         0.248421   
1  coarse  1.000000      0.05     0.492237        0.707232         0.248421   
2  coarse  1.000000      0.10     0.492277        0.707232         0.248421   
3  coarse  1.000000      0.15     0.492458        0.707232         0.248421   
4  coarse  1.778279      0.10     0.493449        0.707232         0.248421   
5  coarse  1.778279      0.15     0.493958        0.707232         0.248421   
6  coarse  0.562341      0.30     0.

In [9]:
# -------------------------
# 5) 2차 local refinement
#    1차 best 주변에서 C, l1_ratio, cutoff 동시 재탐색
# -------------------------
C_grid_2 = best_C_1 * np.logspace(-0.5, 0.5, 15)
l1_min = max(0.0, best_l1_1 - 0.20)
l1_max = min(1.0, best_l1_1 + 0.20)
l1_grid_2 = np.round(np.linspace(l1_min, l1_max, 15), 3)

search_history_2 = []
threshold_history_all_2 = []

print(f"\n[2차 세부 탐색] LR 조합 수: {len(C_grid_2) * len(l1_grid_2)}")
print("각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.")
print("선택 기준: valid G-Mean 최고값의 99% 이상 후보 중 valid Precision 최대")

for C_val in C_grid_2:
    for l1_val in l1_grid_2:
        model_trial = make_lr(C=C_val, max_iter=10000, l1_ratio=l1_val)
        model_trial.fit(X_train_model, y_train)

        valid_prob = model_trial.predict_proba(X_valid_model)[:, 1]
        best_thr_row, threshold_df_tmp = find_best_cutoff(y_valid, valid_prob)

        search_history_2.append({
            "stage": "refine",
            "C": C_val,
            "l1_ratio": l1_val,
            "best_cutoff": best_thr_row["threshold"],
            "valid_accuracy": best_thr_row["valid_accuracy"],
            "valid_precision": best_thr_row["valid_precision"],
            "valid_recall": best_thr_row["valid_recall"],
            "valid_specificity": best_thr_row["valid_specificity"],
            "valid_f1": best_thr_row["valid_f1"],
            "valid_gmean": best_thr_row["valid_gmean"],
            "valid_log_loss": best_thr_row["valid_log_loss"],
            "valid_roc_auc": best_thr_row["valid_roc_auc"],
            "valid_pr_auc": best_thr_row["valid_pr_auc"],
            "tn": best_thr_row["tn"],
            "fp": best_thr_row["fp"],
            "fn": best_thr_row["fn"],
            "tp": best_thr_row["tp"]
        })

        threshold_df_tmp = threshold_df_tmp.copy()
        threshold_df_tmp["stage"] = "refine"
        threshold_df_tmp["C"] = C_val
        threshold_df_tmp["l1_ratio"] = l1_val
        threshold_history_all_2.append(threshold_df_tmp)

search_df_2 = pd.DataFrame(search_history_2)

search_df = pd.concat([search_df_1, search_df_2], ignore_index=True)

max_gmean_all = search_df["valid_gmean"].max()
search_df_eligible = search_df[
    search_df["valid_gmean"] >= 0.99 * max_gmean_all
].copy()

search_df = search_df_eligible.sort_values(
    by=["valid_precision", "valid_gmean", "valid_f1", "valid_recall", "valid_log_loss"],
    ascending=[False, False, False, False, True]
).reset_index(drop=True)

best_row = search_df.iloc[0]
best_lr_params = {
    "C": float(best_row["C"]),
    "l1_ratio": float(best_row["l1_ratio"])
}
best_cutoff = float(best_row["best_cutoff"])

print("\n[최종 Best Hyperparameters + Cutoff]")
print(f"C        : {best_lr_params['C']:.8f}")
print(f"l1_ratio : {best_lr_params['l1_ratio']:.4f}")
print(f"cutoff   : {best_cutoff:.4f}")
print(f"G-Mean   : {best_row['valid_gmean']:.4f}")
print(f"Precision: {best_row['valid_precision']:.4f}")
print(f"F1       : {best_row['valid_f1']:.4f}")
print(f"Recall   : {best_row['valid_recall']:.4f}")
print(f"Specificity: {best_row['valid_specificity']:.4f}")
print(f"LogLoss  : {best_row['valid_log_loss']:.4f}")

print("\n[전체 탐색 상위 15개: G-Mean 99% 이상 후보 중 Precision 우선]")
print(search_df.head(15))

# -------------------------
# 6) 최종 모델 학습
# -------------------------
model = make_lr(
    C=best_lr_params["C"],
    l1_ratio=best_lr_params["l1_ratio"]
)
model.fit(X_train_model, y_train)


[2차 세부 탐색] LR 조합 수: 225
각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.
선택 기준: valid G-Mean 최고값의 99% 이상 후보 중 valid Precision 최대

[최종 Best Hyperparameters + Cutoff]
C        : 0.84834290
l1_ratio : 0.1710
cutoff   : 0.4913
G-Mean   : 0.6812
Precision: 0.2484
F1       : 0.3592
Recall   : 0.6484
Specificity: 0.7158
LogLoss  : 0.6229

[전체 탐색 상위 15개: G-Mean 99% 이상 후보 중 Precision 우선]
     stage         C  l1_ratio  best_cutoff  valid_accuracy  valid_precision  \
0   refine  0.848343     0.171     0.491254        0.707232         0.248421   
1   refine  0.848343     0.186     0.491287        0.707232         0.248421   
2   refine  0.848343     0.200     0.491409        0.707232         0.248421   
3   coarse  1.000000     0.000     0.492223        0.707232         0.248421   
4   refine  1.000000     0.000     0.492223        0.707232         0.248421   
5   refine  1.000000     0.014     0.492212        0.707232         0.248421   
6   refine  1.000000     0.029     0.492222        0.707232       

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.848342898244072
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.171
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- F

### 3. test data 성능 평가


In [10]:
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

X_test_df = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].astype(int)

missing_in_test = set(feature_names) - set(X_test_df.columns)
extra_in_test = set(X_test_df.columns) - set(feature_names)

if missing_in_test or extra_in_test:
    raise ValueError(
        f"train-test 컬럼 불일치\n"
        f"test에 없는 train 컬럼: {missing_in_test}\n"
        f"test에만 있는 컬럼: {extra_in_test}"
    )

X_test_df = X_test_df[feature_names]
X_test = X_test_df.values

# Test 예측
y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_cutoff).astype(int)
test_metrics = get_binary_metrics(y_test, y_test_prob, threshold=best_cutoff)

# Valid 재계산 (Cell 8에서 로드된 valid_df 사용)
y_valid_prob = model.predict_proba(X_valid_model)[:, 1]
y_valid_pred = (y_valid_prob >= best_cutoff).astype(int)
valid_metrics = get_binary_metrics(y_valid, y_valid_prob, threshold=best_cutoff)

# ========== 성능 출력 ==========
print("\n[VALID performance]")
print(f"Cutoff    : {best_cutoff:.4f}")
print(f"Accuracy  : {valid_metrics['accuracy']:.4f}")
print(f"Precision : {valid_metrics['precision']:.4f}")
print(f"Recall    : {valid_metrics['recall']:.4f}")
print(f"Specificity: {valid_metrics['specificity']:.4f}")
print(f"F1-score  : {valid_metrics['f1']:.4f}")
print(f"G-Mean    : {valid_metrics['gmean']:.4f}")
print(f"ROC-AUC   : {valid_metrics['roc_auc']:.4f}")
print(f"PR-AUC    : {valid_metrics['pr_auc']:.4f}")
print(f"LogLoss   : {valid_metrics['log_loss']:.4f}")

print("\n[TEST performance]")
print(f"Cutoff    : {best_cutoff:.4f}")
print(f"Accuracy  : {test_metrics['accuracy']:.4f}")
print(f"Precision : {test_metrics['precision']:.4f}")
print(f"Recall    : {test_metrics['recall']:.4f}")
print(f"Specificity: {test_metrics['specificity']:.4f}")
print(f"F1-score  : {test_metrics['f1']:.4f}")
print(f"G-Mean    : {test_metrics['gmean']:.4f}")
print(f"ROC-AUC   : {test_metrics['roc_auc']:.4f}")
print(f"PR-AUC    : {test_metrics['pr_auc']:.4f}")
print(f"LogLoss   : {test_metrics['log_loss']:.4f}")

print("\n[TEST Confusion Matrix]")
print(confusion_matrix(y_test, y_test_pred, labels=[0, 1]))

print("\n[TEST Classification Report]")
print(classification_report(y_test, y_test_pred, digits=4, zero_division=0))


[VALID performance]
Cutoff    : 0.4913
Accuracy  : 0.7072
Precision : 0.2484
Recall    : 0.6484
Specificity: 0.7158
F1-score  : 0.3592
G-Mean    : 0.6812
ROC-AUC   : 0.7286
PR-AUC    : 0.3294
LogLoss   : 0.6229

[TEST performance]
Cutoff    : 0.4913
Accuracy  : 0.6971
Precision : 0.2283
Recall    : 0.6364
Specificity: 0.7054
F1-score  : 0.3360
G-Mean    : 0.6700
ROC-AUC   : 0.7328
PR-AUC    : 0.3074
LogLoss   : 0.6127

[TEST Confusion Matrix]
[[510 213]
 [ 36  63]]

[TEST Classification Report]
              precision    recall  f1-score   support

           0     0.9341    0.7054    0.8038       723
           1     0.2283    0.6364    0.3360        99

    accuracy                         0.6971       822
   macro avg     0.5812    0.6709    0.5699       822
weighted avg     0.8491    0.6971    0.7474       822



In [11]:
# 정오 행렬 출력

print("\n[VALID Confusion Matrix]")
display(confusion_matrix(y_valid, y_valid_pred, labels=[0, 1]))

print("\n[TEST Confusion Matrix]")
display(confusion_matrix(y_test, y_test_pred, labels=[0, 1]))


[VALID Confusion Matrix]


array([[899, 357],
       [ 64, 118]])


[TEST Confusion Matrix]


array([[510, 213],
       [ 36,  63]])

In [12]:
# ========== 결과 저장 ==========
result_dir = Path("../../results/results_ML")
result_dir.mkdir(parents=True, exist_ok=True)

# valid_df/test_df에 Date가 있으면 그대로 사용
if "Date" in valid_df.columns and "Date" in test_df.columns:
    valid_dates = pd.to_datetime(valid_df["Date"]).dt.strftime("%Y-%m-%d")
    test_dates = pd.to_datetime(test_df["Date"]).dt.strftime("%Y-%m-%d")

# Date가 없으면 data_selected.csv에서 chronological split 기준으로 가져옴
else:
    date_df = pd.read_csv("../../data/processed/data_selected.csv")
    date_df["Date"] = pd.to_datetime(date_df["Date"])
    date_df = date_df.sort_values("Date").reset_index(drop=True)

    n_valid = len(y_valid_pred)
    n_test = len(y_test_pred)

    valid_dates = (
        date_df["Date"]
        .tail(n_valid + n_test)
        .head(n_valid)
        .dt.strftime("%Y-%m-%d")
        .reset_index(drop=True)
    )

    test_dates = (
        date_df["Date"]
        .tail(n_test)
        .dt.strftime("%Y-%m-%d")
        .reset_index(drop=True)
    )

# Date + LR_Pred만 저장
valid_pred_df = pd.DataFrame({
    "Date": valid_dates,
    "LR_Pred": y_valid_pred
})

test_pred_df = pd.DataFrame({
    "Date": test_dates,
    "LR_Pred": y_test_pred
})

valid_pred_df.to_csv(result_dir / "01. LR_valid_pred.csv", index=False, encoding="utf-8-sig")
test_pred_df.to_csv(result_dir / "01. LR_test_pred.csv", index=False, encoding="utf-8-sig")

print('\n✓ "01. LR_valid_pred.csv", "01. LR_test_pred.csv" 파일로 저장 완료')


✓ "01. LR_valid_pred.csv", "01. LR_test_pred.csv" 파일로 저장 완료
